In [ ]:
import os
os.chdir(path_to_wd)

from collections import defaultdict
import pandas as pd
import scanpy as sc
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from glmpca import glmpca
from itertools import combinations
import torch

import sys
from importlib import reload

import gaston
from gaston import neural_net,cluster_plotting, dp_related, segmented_fit, restrict_spots, model_selection
from gaston import binning_and_plotting, isodepth_scaling, run_slurm_scripts, parse_adata
from gaston import spatial_gene_classification, plot_cell_types, filter_genes, process_NN_output

import seaborn as sns
import math

from gaston.cluster_plotting import compute_velocity_on_grid
from gaston.dp_related import rotate_by_theta 

plt.rcParams['font.family']= 'Dejavu serif'
plt.rcParams['pdf.fonttype'] = 'truetype'
matplotlib.rcParams['figure.dpi'] = 100

In [ ]:
data_dir = "Reproducibility/Results/Slide-seqV2/"
os.makedirs(data_dir, exist_ok = True)

save_dir = "Reproducibility/Results/Slide-seqV2/GASTON/BC096/"
os.makedirs(save_dir, exist_ok = True)

## Pre-processing

In [ ]:
st = sc.read("Reproducibility/Results/Slide-seqV2/puck_adata_mc50_BC096.h5ad")

In [ ]:
counts_mat = st.X.toarray()
coords_df = pd.DataFrame({'x': st.obs['x'],'y': st.obs['y']})
coords_mat = coords_df.values
gene_labels = np.array(st.var_names)

data_dir = f'Reproducibility/Results/Slide-seqV2/'
os.makedirs(data_dir, exist_ok = True)

# save matrices
np.save(f'{save_dir}counts_mat.npy', counts_mat)
np.save(f'{save_dir}coords_mat.npy', coords_mat)
np.save(f'{save_dir}gene_labels.npy', gene_labels)

# GLM-PCA parameters
num_dims=5
penalty=100 # may need to increase if this is too small 20, 50 error

# CHANGE THESE PARAMETERS TO REDUCE RUNTIME
num_iters=30
eps=1e-4
num_genes=10000

counts_mat_glmpca=counts_mat[:,np.argsort(np.sum(counts_mat, axis=0))[-num_genes:]]
glmpca_res=glmpca.glmpca(counts_mat_glmpca.T, num_dims, fam="poi", penalty=penalty, verbose=True,
                        ctl = {"maxIter":num_iters, "eps":eps, "optimizeTheta":True})
A = glmpca_res['factors'] # should be of size N x num_dims, where each column is a PC

np.save(f'{save_dir}glmpca.npy', A)

# visualize top GLM-PCs
rotated_coords=dp_related.rotate_by_theta(coords_mat, -np.pi/2)
R=2
C=4
fig,axs=plt.subplots(R,C,figsize=(20,10))
for r in range(R):
    for c in range(C):
        i=r*C+c
        axs[r,c].scatter(rotated_coords[:,0], rotated_coords[:,1], c=A[:,i],cmap='Reds',s=3)
        if i < num_dims:
            axs[r,c].set_title(f'GLM-PC{i}')
        else:
            axs[r,c].set_title('RGB'[i-num_dims])

plt.show()


## Train GASTON neural network

In [ ]:
# Option 1: Slurm (Recommended)
path_to_glmpca=f'{save_dir}glmpca.npy'
path_to_coords=f'{save_dir}coords_mat.npy'

# GASTON NN parameters
isodepth_arch=[20,20] # architecture (two hidden layers of size 20) for isodepth neural network d(x,y)
expression_arch=[20,20] # architecture (two hidden layers of size 20) for 1-D expression function
epochs = 10000 # number of epochs to train NN
checkpoint = 500 # save model after number of epochs = multiple of checkpoint
optimizer = "adam"
num_restarts=30 # number of initializations

# REPLACE with your own conda environment name and path
conda_environment='GASTON'
path_to_conda_folder='/home/k-makino/miniconda3/bin/activate'
partition='raphael' # if you have a separate partition on the cluster

out_dir = f'{save_dir}outputs/'
os.makedirs(out_dir, exist_ok = True)

run_slurm_scripts.train_NN_parallel(path_to_coords, path_to_glmpca, isodepth_arch, expression_arch, 
                      out_dir, conda_environment, path_to_conda_folder,
                      epochs=epochs, checkpoint=checkpoint, 
                      num_seeds=num_restarts, partition=partition
                      )

In [ ]:
WD_DIR="${path_to_wd}"

for seed in {0..29}
do
  "${HOME}/miniconda3/envs/GASTON/bin/gaston" \
    -i "${WD_DIR}/Reproducibility/Results/Slide-seqV2/GASTON/BC096/coords_mat.npy" \
    -o "${WD_DIR}/Reproducibility/Results/Slide-seqV2/GASTON/BC096/glmpca.npy" \
    --epochs 10000 \
    -d "${WD_DIR}/Reproducibility/Results/Slide-seqV2/GASTON/BC096/outputs/" \
    --hidden_spatial 20 20 \
    --hidden_expression 20 20 \
    --optimizer adam \
    --seed "${seed}" \
    -c 500
done

## Process neural network output

In [ ]:
def plot_isodepth2(gaston_isodepth, S, mod, figsize=(5,8), contours=True, contour_levels=4, contour_lw=1, contour_fs=10, colorbar=True, s=20, cbar_fs=10, axis_off=True, streamlines=False, streamlines_lw=1.5, rotate=None, cmap='coolwarm', norm=None,
                 arrowsize=2, neg_gradient=False, scaling_factors=None, gaston_labels_for_scaling=None, linear_transform=None):
    if rotate is not None:
        S_rotated = rotate_by_theta(S, rotate)
    elif linear_transform is not None:
        S_rotated = (linear_transform @ S.T).T
    else:
        S_rotated = S
    fig, ax = plt.subplots(figsize=figsize)
    # Rasterize the scatter plot
    im1 = ax.scatter(S_rotated[:, 0], S_rotated[:, 1], c=gaston_isodepth, cmap=cmap, s=s, norm=norm, rasterized=True)
    if axis_off:
        plt.axis('off')
    if contours:
        CS = ax.tricontour(S_rotated[:, 0], S_rotated[:, 1], gaston_isodepth, levels=contour_levels, linewidths=contour_lw, colors='k', linestyles='solid')
        ax.clabel(CS, CS.levels, inline=True, fontsize=contour_fs)
    if colorbar:
        cbar = plt.colorbar(im1)
        cbar.ax.tick_params(labelsize=cbar_fs)
    if streamlines:
        x = torch.tensor(S, requires_grad=True).float()
        G = torch.autograd.grad(outputs=mod.spatial_embedding(x).flatten(), inputs=x, grad_outputs=torch.ones_like(x[:, 0]))[0]
        G = G.detach().numpy()
        if neg_gradient:
            G = -1 * G
        if scaling_factors is not None:
            L = len(np.unique(gaston_labels_for_scaling))
            for l in range(L):
                G[gaston_labels_for_scaling == l, :] = scaling_factors[l] * G[gaston_labels_for_scaling == l, :]
        if rotate is not None:
            G_rotated = rotate_by_theta(G, rotate)
        elif linear_transform is not None:
            G_rotated = (linear_transform @ G.T).T
        else:
            G_rotated = G
        # CODE FROM scVelo
        smooth = None
        min_mass = None
        n_neighbors = 1000
        cutoff_perc = 0
        X_grid, V_grid = compute_velocity_on_grid(
            X_emb=S_rotated,
            V_emb=G_rotated,
            density=1,
            smooth=smooth,
            min_mass=min_mass,
            n_neighbors=n_neighbors,
            adjust_for_stream=True,
            cutoff_perc=cutoff_perc,
        )
        lengths = np.sqrt((V_grid**2).sum(0))
        linewidth = streamlines_lw
        linewidth *= 2 * lengths / lengths[~np.isnan(lengths)].max()
        density = 1
        stream_kwargs = {
            "linewidth": linewidth,
            "density": density,
            "zorder": 3,
            "color": "k",
            "arrowsize": arrowsize,
            "arrowstyle": "-|>",
            "maxlength": 1000,
            "integration_direction": "both",
        }
        ax.streamplot(X_grid[0], X_grid[1], V_grid[0], V_grid[1], **stream_kwargs)
    ax.set_rasterization_zorder(True)
    plt.axis('off')


In [ ]:
def plot_clusters2(gaston_labels, S, fig=None, ax=None, figsize=(5,8), colors=None, color_palette=plt.cm.Dark2, 
                 s=20,labels=None,
                 lgd=False, rotate=None, show_boundary=False, gaston_isodepth=None, boundary_lw=10, bbox_to_anchor=None,
                 linear_transform=None):
    if rotate is not None:
        S=rotate_by_theta(S,rotate)
    elif linear_transform is not None:
        S=(linear_transform @ S.T).T
    if fig is None or ax is None:
        fig,ax=plt.subplots(figsize=figsize)
    if colors is None:
        colors=np.array([color_palette(i) for i in range(len(np.unique(gaston_labels)))])
    for i,t in enumerate(np.unique(gaston_labels)):
        pts_t=np.where(gaston_labels==t)[0]
        if labels is not None:
            l=labels[i]
        else:
            l=i
        ax.scatter(S[pts_t,0], S[pts_t,1],s=s, color=colors[int(t)], label=l, rasterized=True)
    if show_boundary:
        plt.tricontour(S[:,0], S[:,1], gaston_isodepth, 
               levels=[np.min(gaston_isodepth[gaston_labels==i]) for i in range(len(np.unique(gaston_labels)))[1:]], 
               linewidths=boundary_lw, colors='k', linestyles='-')
    plt.axis('off')
    if lgd:
        plt.legend(bbox_to_anchor=bbox_to_anchor)
    ax.set_rasterization_zorder(True)

In [ ]:
# gaston_model, A, S= process_NN_output.process_files('cerebellum_tutorial_outputs')
gaston_model, A, S= process_NN_output.process_files(out_dir) # TO MATCH PAPER FIGURES

# may need to re-load counts_mat, coords_mat, gene_labels from previous step
counts_mat=np.load(f'{save_dir}counts_mat.npy', allow_pickle=True) # N x G UMI count array
coords_mat=np.load(f'{save_dir}coords_mat.npy', allow_pickle=True) # N x 2 spatial coordinate matrix
gene_labels=np.load(f'{save_dir}gene_labels.npy', allow_pickle=True) # list of names for G genes

# Model selection for choosing number of layers
model_selection.plot_ll_curve(gaston_model, A, S, max_domain_num=8, start_from=2)
plt.show()

# CHANGE FOR YOUR APPLICATION: use number of domains from above!
num_layers=6
gaston_isodepth, gaston_labels=dp_related.get_isodepth_labels(gaston_model,A,S,num_layers)

# DATASET-SPECIFIC: so domains are ordered oligodendrocyte to molecular, with increasing isodepth
gaston_isodepth=np.max(gaston_isodepth) - gaston_isodepth
gaston_labels=(num_layers-1)-gaston_labels

gaston_isodepth=isodepth_scaling.adjust_isodepth(gaston_isodepth, gaston_labels, coords_mat)

# plotting parameters
show_streamlines=False
rotate=np.radians(0)
arrowsize=0

# Plot topographic map: isodepth and spatial gradients
plot_isodepth2(gaston_isodepth, S, gaston_model, figsize=(7,6), contours=False, 
               streamlines=show_streamlines, rotate=rotate, arrowsize=arrowsize, neg_gradient=True)
plt.show()

# Plot GASTON domains
domain_colors=colors=['darksalmon','deepskyblue', 'limegreen' ,'red', 'purple', 'orange']
labels=['A', 'B', 'C', 'D', "E", 'F']
plot_clusters2(gaston_labels, S, figsize=(6,6), colors=domain_colors, 
               s=1, lgd=True, boundary_lw=0, labels = labels,
               show_boundary=True, gaston_isodepth=gaston_isodepth, rotate=rotate,
               bbox_to_anchor=(1.05, 1))
plt.show()

## Spatially varying gene analysis

In [ ]:
isodepth_min=0
isodepth_max=700

cluster_plotting.plot_clusters_restrict(gaston_labels, S, gaston_isodepth, 
                                        isodepth_min=isodepth_min, isodepth_max=isodepth_max, figsize=(6,6), 
                                        colors=domain_colors, s=1, lgd=False, rotate=rotate)
plt.show()

In [ ]:
# Optional: adjust isodepth for physical distance
adjust_physical=True
plot_isodepth=True

# plotting parameters
show_streamlines=True
rotate=np.radians(0)
arrowsize=1

counts_mat_restrict, coords_mat_restrict, gaston_isodepth_restrict, gaston_labels_restrict, S_restrict=restrict_spots.restrict_spots(
                                                             counts_mat, coords_mat, S, gaston_isodepth, gaston_labels, 
                                                             isodepth_min=isodepth_min, isodepth_max=isodepth_max, 
                                                             adjust_physical=adjust_physical, 
                                                             plot_isodepth=plot_isodepth, show_streamlines=show_streamlines, 
                                                             gaston_model=gaston_model, rotate=rotate, figsize=(6,3), 
                                                             arrowsize=arrowsize, 
                                                             neg_gradient=True) # since we reversed gradient direction earlier

# Compute piecewise linear fits
umi_thresh = 20
idx_kept, gene_labels_idx=filter_genes.filter_genes(counts_mat, gene_labels, 
                                       umi_threshold=umi_thresh, 
                                       exclude_prefix=['MT-', 'RPL', 'RPS'])
# compute piecewise linear fit for restricted spots
pw_fit_dict=segmented_fit.pw_linear_fit(counts_mat_restrict, gaston_labels_restrict, gaston_isodepth_restrict,
                                        None, [],  idx_kept=idx_kept, umi_threshold=umi_thresh, isodepth_mult_factor=0.01,)
# for plotting
binning_output=binning_and_plotting.bin_data(counts_mat_restrict, gaston_labels_restrict, gaston_isodepth_restrict, 
                         None, gene_labels, idx_kept=idx_kept, num_bins=15, umi_threshold=umi_thresh)

In [ ]:
q=0.9 # use 0.9 quantile for slopes, discontinuities
discont_genes_layer=spatial_gene_classification.get_discont_genes(pw_fit_dict, binning_output,q=q)
cont_genes_layer=spatial_gene_classification.get_cont_genes(pw_fit_dict, binning_output,q=q)

genes = list(cont_genes_layer.keys())

# Swap first and second columns
coords_mat_restrict2 = coords_mat_restrict.copy()
#coords_mat_restrict2[:, 0] *= -1   # flip x
#coords_mat_restrict2[:, 1] *= -1   # flip y

###################
gene_name = 'CCL21'
binning_and_plotting.plot_gene_function(gene_name, coords_mat_restrict2, pw_fit_dict, gaston_labels_restrict, gaston_isodepth_restrict, 
                                        binning_output, figsize=(5,5),rotate=0,s=1,colorbar=False)
plt.title(f'{gene_name} GASTON Function')
plt.savefig(f'Reproducibility/Results/Plots/Slide-seqV2/FigureS11G_{gene_name}.pdf', bbox_inches = 'tight')
plt.close()

###################
gene_name = 'CCL19'
binning_and_plotting.plot_gene_function(gene_name, coords_mat_restrict2, pw_fit_dict, gaston_labels_restrict, gaston_isodepth_restrict, 
                                        binning_output, figsize=(5,5),rotate=0,s=1,colorbar=False)
plt.title(f'{gene_name} GASTON Function')
plt.savefig(f'Reproducibility/Results/Plots/Slide-seqV2/FigureS11G_{gene_name}.pdf', bbox_inches = 'tight')
plt.close()

###################
gene_name = 'CD79B'
binning_and_plotting.plot_gene_function(gene_name, coords_mat_restrict2, pw_fit_dict, gaston_labels_restrict, gaston_isodepth_restrict, 
                                        binning_output, figsize=(5,5),rotate=0,s=1,colorbar=False)
plt.title(f'{gene_name} GASTON Function')
plt.savefig(f'Reproducibility/Results/Plots/Slide-seqV2/FigureS11G_{gene_name}.pdf', bbox_inches = 'tight')
plt.close()

In [ ]:
tmp = pd.DataFrame(coords_mat_restrict2)
tmp.to_csv("Reproducibility/Results/Slide-seqV2/GASTON/BC096/coords_mat_restrict.txt", sep='\t')